# Day 1 — Data Ingestion & Exploration

This notebook demonstrates the complete Day 1 pipeline:
1. Copy all 10 source CSVs into `data/raw/`
2. Profile each dataset (shape, dtypes, nulls, duplicates, sample rows)
3. Explore `01_fund_master.csv` (unique AMCs, categories, sub-categories, risk grades)
4. Fetch live NAV data from `api.mfapi.in` for targeted AMFI codes
5. Cross-dataset AMFI code validation between fund master and NAV history

In [1]:
import pandas as pd
import numpy as np
import os
import shutil
from pathlib import Path
from datetime import datetime

BASE_DIR = Path().resolve().parent
SOURCE_DIR = BASE_DIR / "data_sets"
RAW_DIR = BASE_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory : {BASE_DIR}")
print(f"Source datasets: {SOURCE_DIR}")
print(f"Raw output     : {RAW_DIR}")

Base directory : C:\Users\Dell\Desktop\Internship_coding
Source datasets: C:\Users\Dell\Desktop\Internship_coding\data_sets
Raw output     : C:\Users\Dell\Desktop\Internship_coding\data\raw


## Step 1 — Copy Source CSVs to `data/raw/`

In [2]:
csv_files = sorted(SOURCE_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} source CSV files:\n")

for src in csv_files:
    dst = RAW_DIR / src.name
    if not dst.exists():
        shutil.copy(src, dst)
        print(f"  ✅ Copied {src.name}")
    else:
        print(f"  ⏭️  {src.name} already exists in data/raw/")

Found 10 source CSV files:

  ⏭️  01_fund_master.csv already exists in data/raw/
  ⏭️  02_nav_history.csv already exists in data/raw/
  ⏭️  03_aum_by_fund_house.csv already exists in data/raw/
  ⏭️  04_monthly_sip_inflows.csv already exists in data/raw/
  ⏭️  05_category_inflows.csv already exists in data/raw/
  ⏭️  06_industry_folio_count.csv already exists in data/raw/
  ⏭️  07_scheme_performance.csv already exists in data/raw/
  ⏭️  08_investor_transactions.csv already exists in data/raw/
  ⏭️  09_portfolio_holdings.csv already exists in data/raw/
  ⏭️  10_benchmark_indices.csv already exists in data/raw/


## Step 2 — Profile All 10 Datasets

For each CSV we print: **shape**, **column dtypes**, **null counts**, **duplicate rows**, and **sample rows (head)**.

In [3]:
raw_files = sorted(RAW_DIR.glob("[0-9]*.csv"))

profile_summary = []

for fp in raw_files:
    df = pd.read_csv(fp)
    nulls = df.isnull().sum()
    total_nulls = nulls.sum()
    dups = df.duplicated().sum()

    profile_summary.append({
        "Dataset": fp.name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Nulls": total_nulls,
        "Duplicates": dups,
    })

    print(f"{'=' * 70}")
    print(f"📄 {fp.name}")
    print(f"{'=' * 70}")
    print(f"  Shape       : {df.shape}")
    print(f"  Columns     : {list(df.columns)}")
    print(f"  Total Nulls : {total_nulls}")
    print(f"  Duplicates  : {dups}")
    if total_nulls > 0:
        print(f"  Null detail : {nulls[nulls > 0].to_dict()}")
    print(f"\n  Dtypes:")
    for col, dtype in df.dtypes.items():
        print(f"    {col:40s} {dtype}")
    print(f"\n  Head (5 rows):")
    display(df.head())
    print()

📄 01_fund_master.csv
  Shape       : (40, 15)
  Columns     : ['amfi_code', 'fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'launch_date', 'benchmark', 'expense_ratio_pct', 'exit_load_pct', 'min_sip_amount', 'min_lumpsum_amount', 'fund_manager', 'risk_category', 'sebi_category_code']
  Total Nulls : 0
  Duplicates  : 0

  Dtypes:
    amfi_code                                int64
    fund_house                               object
    scheme_name                              object
    category                                 object
    sub_category                             object
    plan                                     object
    launch_date                              object
    benchmark                                object
    expense_ratio_pct                        float64
    exit_load_pct                            float64
    min_sip_amount                           int64
    min_lumpsum_amount                       int64
    fund_manager            

,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


📄 02_nav_history.csv
  Shape       : (46000, 3)
  Columns     : ['amfi_code', 'date', 'nav']
  Total Nulls : 0
  Duplicates  : 0

  Dtypes:
    amfi_code                                int64
    date                                     object
    nav                                      float64

  Head (5 rows):


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692



📄 03_aum_by_fund_house.csv
  Shape       : (90, 5)
  Columns     : ['date', 'fund_house', 'aum_lakh_crore', 'aum_crore', 'num_schemes']
  Total Nulls : 0
  Duplicates  : 0

  Dtypes:
    date                                     object
    fund_house                               object
    aum_lakh_crore                           float64
    aum_crore                                int64
    num_schemes                              int64

  Head (5 rows):


...

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending



📄 09_portfolio_holdings.csv
  Shape       : (322, 8)
  Columns     : ['amfi_code', 'stock_symbol', 'stock_name', 'sector', 'weight_pct', 'market_value_cr', 'current_price_inr', 'portfolio_date']
  Total Nulls : 0
  Duplicates  : 0

  Dtypes:
    amfi_code                                int64
    stock_symbol                             object
    stock_name                               object
    sector                                   object
    weight_pct                               float64
    market_value_cr                          float64
    current_price_inr                        float64
    portfolio_date                           object

  Head (5 rows):


,amfi_code,stock_symbol,stock_name,sector,weight_pct,market_value_cr,current_price_inr,portfolio_date
0,119551,POWERGRID,Power Grid Corporation,Utilities,13.85,737.09,6011.08,2025-12-31
1,119551,HDFCBANK,HDFC Bank Ltd,Banking,11.19,88.97,1074.65,2025-12-31
2,119551,GRASIM,Grasim Industries Ltd,Diversified,9.90,208.45,5964.59,2025-12-31
3,119551,DRREDDY,Dr. Reddy's Laboratories,Pharma,4.76,161.32,3748.82,2025-12-31
4,119551,ASIANPAINT,Asian Paints Ltd,Paints,10.25,725.90,1321.45,2025-12-31



📄 10_benchmark_indices.csv
  Shape       : (8050, 3)
  Columns     : ['date', 'index_name', 'close_value']
  Total Nulls : 0
  Duplicates  : 0

  Dtypes:
    date                                     object
    index_name                               object
    close_value                              float64

  Head (5 rows):


,date,index_name,close_value
0,2022-01-03,NIFTY50,17492.79
1,2022-01-04,NIFTY50,17689.64
2,2022-01-05,NIFTY50,17835.05
3,2022-01-06,NIFTY50,17878.51
4,2022-01-07,NIFTY50,17759.15


In [4]:
# Summary table
summary_df = pd.DataFrame(profile_summary)
print("\n📊 INGESTION PROFILE SUMMARY")
print("=" * 70)
display(summary_df)
print(f"\nTotal records across all datasets: {summary_df['Rows'].sum():,}")
print(f"Total null values: {summary_df['Nulls'].sum():,}")
print(f"Total duplicates:  {summary_df['Duplicates'].sum():,}")


📊 INGESTION PROFILE SUMMARY


,Dataset,Rows,Columns,Nulls,Duplicates
0,01_fund_master.csv,40,15,0,0
1,02_nav_history.csv,46000,3,0,0
2,03_aum_by_fund_house.csv,90,5,0,0
3,04_monthly_sip_inflows.csv,48,6,12,0
4,05_category_inflows.csv,144,3,0,0
5,06_industry_folio_count.csv,21,6,0,0
6,07_scheme_performance.csv,40,19,0,0
7,08_investor_transactions.csv,32778,13,0,0
8,09_portfolio_holdings.csv,322,8,0,0
9,10_benchmark_indices.csv,8050,3,0,0



Total records across all datasets: 87,533
Total null values: 12
Total duplicates:  0


## Step 3 — Fund Master Exploration

Extract unique fund houses, categories, sub-categories, and risk grades from `01_fund_master.csv`.

In [5]:
master = pd.read_csv(RAW_DIR / "01_fund_master.csv")
print(f"Total Schemes: {len(master)}")
display(master.head())

Total Schemes: 40


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03
3,119599,SBI Mutual Fund,SBI Small Cap Fund - Direct Plan - Growth,Equity,Small Cap,Direct,2013-01-01,BSE 250 SmallCap TRI,0.72,1.0,500,1000,R. Srinivasan,Very High,EC03
4,119120,SBI Mutual Fund,SBI Magnum Gilt Fund - Regular Plan - Growth,Debt,Gilt,Regular,2000-12-30,CRISIL Dynamic Gilt Index,0.77,0.0,500,1000,Dinesh Ahuja,Low,DC02


In [6]:
print(f"\n🏢 Unique Fund Houses ({master['fund_house'].nunique()}):")
for fh in sorted(master['fund_house'].unique()):
    print(f"  • {fh}")

print(f"\n📁 Unique Categories ({master['category'].nunique()}):")
for cat in sorted(master['category'].unique()):
    print(f"  • {cat}")

print(f"\n📂 Unique Sub-Categories ({master['sub_category'].nunique()}):")
for sub in sorted(master['sub_category'].unique()):
    print(f"  • {sub}")

print(f"\n⚠️ Unique Risk Categories ({master['risk_category'].nunique()}):")
for rc in sorted(master['risk_category'].unique()):
    print(f"  • {rc}")

print(f"\nTotal AMFI Codes: {master['amfi_code'].nunique()}")


🏢 Unique Fund Houses (10):
  • Aditya Birla Sun Life MF
  • Axis Mutual Fund
  • DSP Mutual Fund
  • HDFC Mutual Fund
  • ICICI Prudential MF
  • Kotak Mahindra MF
  • Mirae Asset MF
  • Nippon India MF
  • SBI Mutual Fund
  • UTI Mutual Fund

📁 Unique Categories (2):
  • Debt
  • Equity

📂 Unique Sub-Categories (12):
  • ELSS
  • Flexi Cap
  • Gilt
  • Index
  • Index/ETF
  • Large & Mid Cap
  • Large Cap
  • Liquid
  • Mid Cap
  • Short Duration
  • Small Cap
  • Value

⚠️ Unique Risk Categories (5):
  • High
  • Low
  • Moderate
  • Moderately High
  • Very High

Total AMFI Codes: 40


## Step 4 — Live NAV Fetch from mfapi.in

Fetch live NAV data for 6 target mutual fund schemes via the public `api.mfapi.in` REST API.

In [7]:
import requests

TARGETS = {
    "125497": "HDFC_Top_100_Direct",
    "119551": "SBI_Bluechip",
    "120503": "ICICI_Bluechip",
    "118632": "Nippon_Large_Cap",
    "119092": "Axis_Bluechip",
    "120841": "Kotak_Bluechip",
}

for amfi_code, label in TARGETS.items():
    url = f"https://api.mfapi.in/mf/{amfi_code}"
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        nav_data = data.get("data", [])
        if nav_data:
            df_nav = pd.DataFrame(nav_data)
            out_path = RAW_DIR / f"raw_nav_{amfi_code}.csv"
            df_nav.to_csv(out_path, index=False)
            print(f"  ✅ {label} ({amfi_code}): {len(df_nav)} NAV records saved")
        else:
            print(f"  ⚠️  {label} ({amfi_code}): No NAV data returned")
    except Exception as e:
        print(f"  ❌ {label} ({amfi_code}): {e}")

  ✅ HDFC_Top_100_Direct (125497): 3117 NAV records saved


  ✅ SBI_Bluechip (119551): 3262 NAV records saved


  ✅ ICICI_Bluechip (120503): 3333 NAV records saved


  ✅ Nippon_Large_Cap (118632): 3324 NAV records saved


  ✅ Axis_Bluechip (119092): 3591 NAV records saved


  ✅ Kotak_Bluechip (120841): 3327 NAV records saved


## Step 5 — AMFI Code Cross-Validation

Verify that all AMFI codes in the NAV history match those in the fund master.

In [8]:
nav = pd.read_csv(RAW_DIR / "02_nav_history.csv")

master_codes = set(master['amfi_code'].astype(str).unique())
nav_codes = set(nav['amfi_code'].astype(str).unique())

in_master_not_nav = master_codes - nav_codes
in_nav_not_master = nav_codes - master_codes

print(f"AMFI codes in fund_master : {len(master_codes)}")
print(f"AMFI codes in nav_history : {len(nav_codes)}")
print(f"In master but NOT in NAV  : {len(in_master_not_nav)} → {in_master_not_nav or 'None'}")
print(f"In NAV but NOT in master  : {len(in_nav_not_master)} → {in_nav_not_master or 'None'}")

if not in_master_not_nav and not in_nav_not_master:
    print("\n✅ AMFI code integrity check PASSED — all codes match.")
else:
    print("\n⚠️  AMFI code mismatch detected — review before loading into DB.")

AMFI codes in fund_master : 40
AMFI codes in nav_history : 40
In master but NOT in NAV  : 0 → None
In NAV but NOT in master  : 0 → None

✅ AMFI code integrity check PASSED — all codes match.


---
*Day 1 Data Ingestion complete. All 10 datasets profiled, live NAV data fetched, and AMFI code integrity validated.*